In [1]:
# --------------------------------------------------
# Set up project paths and enable auto-reload
# --------------------------------------------------

from pathlib import Path
from sklearn.preprocessing import LabelEncoder
import sys
import numpy as np
import pandas as pd

# --------------------------------------------------
# Paths
# --------------------------------------------------

project_root = Path(r"C:\Users\user\Desktop\multiomic_vae")
data_path = project_root / "processed_data" / "pbmc_10k"

# Add src to Python path
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# --------------------------------------------------
# Auto-reload .py files (Jupyter only)
# --------------------------------------------------

%load_ext autoreload
%autoreload 2

# --------------------------------------------------
# Convenience helper
# --------------------------------------------------

def data_file(name):
    return to_train_data_path / name

print("Project root:", project_root)
print("Data to train:", data_path)


Project root: C:\Users\user\Desktop\multiomic_vae
Data to train: C:\Users\user\Desktop\multiomic_vae\processed_data\pbmc_10k


#### Loading Peak Matrix

In [2]:
from multiomic_vae.utils.train_data_loader import load_npz_as_df
peak_data_path = data_path / "cell_peak_preprocessed.npz"
peak = load_npz_as_df(peak_data_path)

In [3]:
peak.head()

,chr1:629395-630394,chr1:633578-634591,chr1:778283-779200,chr1:816873-817775,chr1:827067-827949,chr1:844152-845011,chr1:869477-870378,chr1:904364-905213,chr1:920762-921638,chr1:923415-924300,...,chrY:14340444-14341328,chrY:14344648-14345541,chrY:15455739-15456657,chrY:19077075-19078016,chrY:19567013-19567787,chrY:19744368-19745303,chrY:20575244-20576162,chrY:21174458-21175369,chrY:21240021-21240909,chrY:21249202-21250057
AAACAGCCACCGGCTA-1,0,1.098612,1.609438,0,0,0,1.098612,0,0,0,...,1.098612,0,0.693147,0,0,1.098612,0,0,0,0
AAACATGCAGGACCAA-1,0,1.098612,0,0,0,0,1.098612,0,0,0,...,0,0,0,0,0,0,1.098612,0,0,0
AAACATGCATGCATAT-1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AAACCAACAGGTTTGC-1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
AAACCGAAGGACCGCT-1,0,1.098612,0.693147,0,0,0,0,0,0,0,...,0,0,0,0,0,1.098612,0,0,0,0


#### Load Gene Matrix

In [4]:
gene_data_path = data_path / "cell_gene_preprocessed.npz"
gene = load_npz_as_df(gene_data_path)

In [5]:
gene.head()

,AL627309.1,AL627309.5,AL627309.4,LINC01409,FAM87B,LINC01128,LINC00115,FAM41C,AL645608.6,SAMD11,...,MT-CYB,BX004987.1,AC145212.1,MAFIP,AC011043.1,AL354822.1,AL592183.1,AC240274.1,AC004556.3,AC007325.4
AAACAGCCACCGGCTA-1,0,0,0,0,0,0,0,0,0,0,...,2.848636,0,0,0,0,0,0,0.734424,0,0
AAACATGCAGGACCAA-1,0,0,0,0,0,0,0.650402,0.650402,0,0,...,2.119922,0,0,0,0,0,0.650402,0.650402,0,0
AAACATGCATGCATAT-1,0,0,0,0,0,0,0,0,0,0,...,2.676348,0,0,0,0,0,0,0,0,0
AAACCAACAGGTTTGC-1,0,0,0,0,0,0,0,0,0,0,...,3.111176,0,0,0,0,0,0,0,0,0
AAACCGAAGGACCGCT-1,0,0,0,0,0,0,0,0,0,0,...,2.238282,0,0,0,0,0,0,0,0,0


#### Wandb

In [6]:
pip install wandb

Note: you may need to restart the kernel to use updated packages.


In [7]:
import wandb
wandb.login(key="1e6e3610162a564623e4b101712cb16ccc2467cc")

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\user\_netrc
wandb: Currently logged in as: farhadyousefi_vae (farhadyousefi_vae_polito) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

#### Using the model

In [8]:
# --------------------------------------------------
# Load + align Leiden labels
# --------------------------------------------------
leiden_path = data_path / "rna_leiden_labels.csv"
leiden_df = pd.read_csv(leiden_path)
leiden_df.head()

,cell_id,leiden
0,AAACAGCCACCGGCTA-1,2
1,AAACATGCAGGACCAA-1,6
2,AAACATGCATGCATAT-1,7
3,AAACCAACAGGTTTGC-1,7
4,AAACCGAAGGACCGCT-1,8


In [9]:
from multiomic_vae.training.gpg_train import gpg_train

model, gene_x, peak_x = gpg_train(gene, peak)

In [10]:
%run ../multiomic_vae/training/gpg_train.py

In [11]:

from multiomic_vae.evaluation.gpg_test import run_tests_with_wandb

C:\Users\user\anaconda3\envs\peakvi\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
from sklearn.preprocessing import LabelEncoder

# If cell IDs are a column, move them to index
if "cell_id" in leiden_df.columns:
    leiden_df = leiden_df.set_index("cell_id")

# Align to gene dataframe order
leiden_df_aligned = leiden_df.loc[gene.index]

# Encode cluster labels as integers
labels = LabelEncoder().fit_transform(
    leiden_df_aligned.iloc[:, 0].astype(str).values
)


In [13]:
print((leiden_df_aligned.index == gene.index).all())


True


In [14]:
import numpy as np

model.eval()

# adapter (NO CHANGE TO OLD FILES)
@torch.no_grad()
def enc1_mu(x):
    return model.mu1(model.enc1(x))

@torch.no_grad()
def enc2_mu(x):
    return model.mu2(model.enc2(x))

@torch.no_grad()
def gene_to_peak(x):
    z1 = model.mu1(model.enc1(x))
    return model.dec1(z1)

@torch.no_grad()
def cycle_gene(x):
    z1 = model.mu1(model.enc1(x))
    peak_hat = model.dec1(z1)
    z2 = model.mu2(model.enc2(peak_hat))
    return model.dec2(z2)

model.enc1_mu = enc1_mu
model.enc2_mu = enc2_mu
model.gene_to_peak = gene_to_peak
model.cycle_gene = cycle_gene

results = run_tests_with_wandb(
    model=model,
    gene_tensor=gene_x,
    peak_tensor=peak_x,
    labels=labels
)

C:\Users\user\anaconda3\envs\peakvi\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\user\anaconda3\envs\peakvi\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [15]:
wandb.finish()


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


beta,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇███████████
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
loss/gene_recon,█▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
loss/kl1,▁▃▆█▇▆▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
loss/kl2,▁█▆▅▅▄▄▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss/peak_recon,█▅▃▂▁▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
loss/total,█▅▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
test/gene_mean_pearson,▁
test/peak_best_threshold,▁
test/peak_f1,▁
+14,...
